# Class 4 practice — Summarizing Data (Pennsylvania preschool market)

Follows [class4-slides.md](class4-slides.md). Run this on the Yale HPC (Open OnDemand ->
JupyterLab), from your home directory.

Today's skills: exploring **missing data** (and *why* something is missing, not just that it
is), handling **outliers**, and **merging** two datasets safely (supply-side preschool data +
demand-side Census data).

Every code cell below is commented line-by-line explaining *what* it does and *why* -- that's
on purpose, this is a learning notebook, not just a working one.

## 0. Load the two datasets

We need **two separate files** for today: the supply side (individual preschools) and the
demand side (Census block-group data -- neighborhood-level population/income stats).

In [ ]:
import pandas as pd   # the main library for working with tables of data
import os              # lets us build file paths piece by piece, instead of typing one long string

# Build the path to the course's shared data folder, piece by piece.
# os.path.join is safer than writing "/gpfs/project/esai_2026/data/preschools" by hand --
# it won't matter if a slash is missing, and it's easier to read.
projectPath = "/gpfs/project/esai_2026/"
dataPath = os.path.join(projectPath, "data", "preschools")
print(dataPath)  # sanity check: does this look like the path you expect?

# Load the supply side: one row per preschool per year.
df_preschools = pd.read_csv(os.path.join(dataPath, "data_preschools.csv"))

# Load the demand side: one row per Census block-group per year (neighborhood-level stats).
df_blockgroup = pd.read_csv(os.path.join(dataPath, "data_blockgroup_2010_2018.csv"))

# Always print the shape (rows, columns) right after loading -- a cheap sanity check that
# catches an obviously wrong file before you waste time analyzing it.
print("preschools:", df_preschools.shape)
print("blockgroup:", df_blockgroup.shape)

In [ ]:
# .info() is a quick first look at any dataframe: every column's name, how many non-missing
# values it has, and its data type (number, text, etc). Comparing the "Non-Null Count" across
# columns is the fastest way to spot which columns have missing data at all.
df_preschools.info()

## 1. Missing data -- not just *that* it's missing, but *why*

Three ways data can be missing, and it matters which one you're dealing with:
- **MCAR** (Missing Completely At Random): pure chance. The data you *do* have is still
  representative -- safe to just drop the missing rows.
- **MAR** (Missing At Random): missingness depends on something you *can* observe (e.g. missing
  more often in early years). You can fix this with imputation.
- **MNAR** (Missing Not At Random): missingness depends on the *value itself* -- e.g. high
  earners not reporting income. Dropping or imputing naively will bias your results; you need a
  model of the missingness itself.

Below we check *where* the missingness concentrates (by county, year, rating) to figure out
which of the three we're likely facing.

In [ ]:
# Create a new column that's just True/False (1/0) for "was enrollment missing on this row?"
# .isna() checks each value for missing/blank; .astype(int) turns True/False into 1/0 so we
# can average it below (the average of a bunch of 0s and 1s is just "the share that are 1").
df_preschools["ind_enrol_miss"] = (
    df_preschools["enrollment_raw"].isna().astype(int)
)

# groupby("county_fips") splits the data into one group per county, then ["ind_enrol_miss"].mean()
# computes the average of our 0/1 flag *within each group* -- i.e. "what fraction of rows in this
# county are missing enrollment?" Same idea for year and rating below.
print("--- missing enrollment share, by county ---")
display(df_preschools.groupby("county_fips")["ind_enrol_miss"].mean())

print("--- missing enrollment share, by year ---")
display(df_preschools.groupby("year")["ind_enrol_miss"].mean())

print("--- missing enrollment share, by STAR rating ---")
display(df_preschools.groupby("stars_rating")["ind_enrol_miss"].mean())

**What to look for**: if missingness is roughly flat across counties/years/ratings, that leans
MCAR. If it's concentrated in specific years, that's MAR (you can see and control for *when*).
If it's concentrated in specific *ratings* (e.g. always missing for STAR 1), that's closer to
MNAR-flavored -- the missingness is tied to the very thing (quality) you're trying to study,
which is exactly what the professor's slides flagged for this dataset: STAR 1 centers aren't
required to report enrollment at all.

## 2. Outliers -- a single extreme value can wreck an average

One huge/tiny value can drag a mean (and any relationship built on it) way off -- but you can't
just delete anything extreme, because sometimes the extreme value is real and informative. The
safer default: **winsorize** (clip values to a reasonable range) instead of deleting rows.

In [ ]:
# .describe() normally gives you count/mean/min/max/25%/50%/75%. Passing `percentiles=` lets us
# ask for specific cut points instead -- here the very extreme 0.5% and 99.5% tails, which is
# where outliers would show up that the default 25/50/75% view wouldn't catch.
df_preschools["price_clean_real2015"].describe(percentiles=[0.005, 0.25, 0.50, 0.75, 0.995])

In [ ]:
# Grab the price column once so we don't retype the long name repeatedly.
col = df_preschools["price_clean_real2015"]

# .quantile([0.005, 0.995]) finds the actual price values at the 0.5th and 99.5th percentile --
# i.e. "the price below which the cheapest 0.5% of centers fall" and the equivalent for the top.
lower, upper = col.quantile([0.005, 0.995])

# .clip(lower=..., upper=...) doesn't delete anything -- it just pulls any value below `lower`
# up to `lower`, and any value above `upper` down to `upper`. Everything in between is untouched.
# This keeps every row in the dataset (no lost sample size) while capping the extreme influence
# of a handful of implausible prices.
df_preschools["price_clean_real2015_win"] = col.clip(lower=lower, upper=upper)

print(f"clipped prices below ${lower:.2f} or above ${upper:.2f}")

In [ ]:
# Enrollment is different from price: it has a *real, known* upper bound -- a center physically
# cannot enroll more kids than its licensed capacity. So instead of guessing a percentile cutoff,
# we use capacity itself as the ceiling. That's a more defensible choice than an arbitrary number.

display(df_preschools["enrollment_clean"].describe(percentiles=[0.005, 0.25, 0.50, 0.75, 0.995]))

# How often does enrollment actually exceed capacity? (Should be rare -- if it's common, that's
# a sign of a data problem, not just a few outliers.)
display((df_preschools["enrollment_clean"] > df_preschools["capacity_clean"]).mean())

col = df_preschools["enrollment_clean"]
lower = col.quantile(0.005)  # still use a small percentile floor for the low end
# For the ceiling, use each row's OWN capacity value (not one fixed number for everyone) --
# note `upper=` here is a whole column, not a single number, so each row gets its own cap.
df_preschools["enrollment_clean_winCap"] = col.clip(
    lower=lower, upper=df_preschools["capacity_clean"],
)

## 3. Descriptives I -- preschools by quality and over time

Simple question: are centers getting better quality over the years, and do higher-rated centers
look different (bigger, pricier)?

In [ ]:
# Keep only rows from 2010 or 2018 (the earliest and latest years) so we can compare the two
# snapshots directly. .isin([...]) checks each row's year against our list of two years.
print("--- share of each STAR rating, 2010 vs 2018 ---")
display(
    df_preschools.loc[df_preschools["year"].isin([2010, 2018])]
    .groupby("year")["stars_rating"]
    .value_counts(normalize=True)   # normalize=True gives PROPORTIONS (sum to 1) not raw counts
    .unstack("year"))                # reshape so years become columns, ratings become rows -- easier to read side by side

# Now: among 2018 centers only, what's the average price/enrollment/capacity/accreditation
# FOR EACH rating? This tells us how much "better" (bigger, pricier) a higher rating really is.
print("--- average characteristics in 2018, by STAR rating ---")
display(
    df_preschools.loc[df_preschools["year"] == 2018]
    .groupby("stars_rating")[["price_clean_real2015",
        "enrollment_clean", "capacity_clean", "ind_accredited"]]
    .mean())

## 4. Descriptives II -- what kind of neighborhood has high preschool enrollment?

Switching to the demand-side (`df_blockgroup`) data: which neighborhoods actually send more of
their kids to preschool?

In [ ]:
# Enrollment SHARE = (kids enrolled in preschool) / (kids of preschool age, 3-4 years old).
# We can't just divide directly -- if the denominator (age 3-4 population) is 0 or missing,
# dividing by it would either crash or produce a nonsense "infinity" value.
denom = df_blockgroup["ct_age_3_and_4"]

# denom.where(denom > 0) keeps the denominator ONLY where it's a real positive number, and
# replaces it with a blank (NaN) everywhere else. Dividing by a blank safely produces a blank
# result too, instead of an error or a fake infinite share.
df_blockgroup["sh_enrol_preschool"] = (
    df_blockgroup["ct_enrol_prschl"] / denom.where(denom > 0))

# pd.qcut splits a continuous number into equal-sized GROUPS based on its value -- here, 3
# groups ("terciles"): the third of neighborhoods with the lowest enrollment share, the middle
# third, and the highest third. Much easier to compare 3 labeled groups than one continuous number.
df_blockgroup["enrol_tercile"] = pd.qcut(
    df_blockgroup["sh_enrol_preschool"], q=3, labels=["Low", "Middle", "High"])

# Now: for each of those three groups, what's the average household income, home value, rent,
# and share of college graduates? This tells us what kind of neighborhood "High" enrollment means.
display(df_blockgroup.groupby("enrol_tercile", observed=True)[["median_hhinc",
    "median_value", "median_rent", "ratio_clg"]].mean())

## 5. Descriptives III -- merging supply (preschools) with demand (neighborhoods)

This is the riskiest step of the day: combining two different datasets into one. Merges can fail
silently -- accidentally duplicating rows, or dropping rows that should have matched -- so every
step here includes a check that confirms the merge did what we expected, not just that it ran
without an error.

In [ ]:
# Step 1: the blockgroup data's "year" is stored as a range like "2006-2010" (a 5-year Census
# estimate window). We only want the LAST year of that range as a single number, so we can match
# it to the preschool data's single-year "year" column.
# .str.split("-") breaks "2006-2010" into ["2006", "2010"]; .str[-1] grabs the last piece ("2010");
# .astype(int) converts that text "2010" into the actual number 2010.
df_blockgroup["year"] = df_blockgroup["YEAR"].str.split("-").str[-1].astype(int)

# Step 2: BEFORE merging, confirm each dataset's "identifier + year" combination is unique --
# i.e. there's only ONE row per preschool per year, and only ONE row per neighborhood per year.
# If that's not true, a merge could silently multiply rows (e.g. 1 preschool row becoming 3).
# `assert` stops the notebook immediately with an error message if the condition is False --
# much better than continuing on bad data without noticing.
assert not df_preschools.duplicated(["mpi_clean", "year"]).any(), (
    "Duplicate mpi_clean/year combinations in df_preschools.")
assert not df_blockgroup.duplicated(["GISJOIN", "year"]).any(), (
    "Duplicate GISJOIN/year combinations in df_blockgroup.")
print("Both datasets have unique identifier/year combinations.")

In [ ]:
# Step 3: pick exactly which neighborhood-level columns we want to attach to each preschool row.
bg_cols = ["median_rent", "median_hhinc", "population_total", "ratio_clg",
           "ct_age_under_5", "ct_age_3_and_4", "ct_enrol_prschl"]

# .drop(columns=..., errors="ignore") removes these columns first if they're already there --
# this makes the cell SAFE TO RE-RUN. Without this, running the merge twice would create
# duplicate/renamed columns instead of cleanly replacing them.
df_preschools = df_preschools.drop(
    columns=bg_cols+["_bg_merge"], errors="ignore").merge(
    df_blockgroup[["GISJOIN", "year"]+bg_cols],  # only bring over the columns we listed above
    on=["GISJOIN", "year"],       # match rows where BOTH the location id AND year agree
    how="left",                    # keep every preschool row, even if no neighborhood match is found
    validate="many_to_one",        # SAFETY CHECK: many preschools can share one neighborhood, but
                                    # each neighborhood/year should match to only one row on the right side.
                                    # If that's violated, this line raises an error instead of silently
                                    # duplicating preschool rows.
    indicator="_bg_merge")         # adds a column recording whether each row matched, for the check below

# Step 4: actually look at how many rows matched vs. didn't, rather than assuming the merge worked.
print(df_preschools["_bg_merge"].value_counts())

# Now that we've checked it, drop the indicator column -- it was only there to help us verify.
df_preschools = df_preschools.drop(columns="_bg_merge")

In [ ]:
# Now that supply (price) and demand (neighborhood income/education) live in the same table,
# we can ask: do pricier preschools sit in richer neighborhoods?

# Same tercile trick as before, applied to preschool price this time.
df_preschools["price_tercile"] = pd.qcut(
    df_preschools["price_clean_real2015"], q=3, labels=["Low", "Middle", "High"])

print("--- neighborhood characteristics, by preschool PRICE tercile ---")
display(df_preschools.groupby("price_tercile", observed=True)[["median_hhinc",
    "median_rent", "ratio_clg"]].mean())

print("--- neighborhood characteristics, by ACCREDITATION status ---")
display(df_preschools.groupby("ind_accredited", observed=True)[["median_hhinc",
    "median_rent", "ratio_clg"]].mean())

**Expected finding** (per the slides): high-price, high-quality/accredited centers cluster in
richer, higher-rent, more-educated neighborhoods -- the supply side telling the same story
Descriptives II told from the demand side.